In [3]:
"""
Task 1: Vectorized Scaled Dot-Product Attention from Scratch
==============================================================

Implements:
    Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) + mask ) V

Requirements covered:
    - Pure NumPy, no high-level DL wrappers (no torch/tf/keras).
    - Batch-wise processing            -> leading axis 0
    - Multi-head processing            -> axis 1
    - 4D tensor shape                  -> (batch, num_heads, seq_len, d_k)
    - Causal (decoder-style) masking
    - Numerically stable softmax
"""

import numpy as np


# ----------------------------------------------------------------------
# 1. Numerically stable softmax (vectorized over the last axis)
# ----------------------------------------------------------------------
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.

    Subtracting the row-wise max before exponentiating prevents overflow
    (exp of a large positive number) without changing the result, since
    softmax(x) == softmax(x - c) for any constant c along that axis.
    """
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)


# ----------------------------------------------------------------------
# 2. Causal mask generator (for decoder self-attention)
# ----------------------------------------------------------------------
def generate_causal_mask(seq_len: int) -> np.ndarray:
    """
    Builds an additive causal mask of shape (seq_len, seq_len).

    Position i (query) is allowed to attend to position j (key) only if
    j <= i. Disallowed positions get -inf (added to the scores before
    softmax, driving their post-softmax probability to 0). Allowed
    positions get 0 (no change to the score).

    This mask broadcasts cleanly against a (batch, heads, seq_len, seq_len)
    score tensor.
    """
    # upper-triangular part (excluding the diagonal) = future positions
    future_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    mask = np.zeros((seq_len, seq_len), dtype=np.float32)
    mask[future_mask] = -np.inf
    return mask


# ----------------------------------------------------------------------
# 3. Core scaled dot-product attention (fully vectorized, 4D)
# ----------------------------------------------------------------------
def scaled_dot_product_attention(Q: np.ndarray,
                                  K: np.ndarray,
                                  V: np.ndarray,
                                  mask: np.ndarray = None,
                                  return_weights: bool = False):
    """
    Vectorized scaled dot-product attention.

    Parameters
    ----------
    Q : np.ndarray, shape (batch, num_heads, seq_len_q, d_k)
    K : np.ndarray, shape (batch, num_heads, seq_len_k, d_k)
    V : np.ndarray, shape (batch, num_heads, seq_len_k, d_v)
    mask : np.ndarray or None
        Additive mask broadcastable to (batch, num_heads, seq_len_q, seq_len_k).
        Use 0 for "keep" and -np.inf for "block" (e.g. from
        generate_causal_mask). Padding masks can be passed the same way.
    return_weights : bool
        If True, also return the attention-weight matrix.

    Returns
    -------
    output : np.ndarray, shape (batch, num_heads, seq_len_q, d_v)
    weights : np.ndarray, shape (batch, num_heads, seq_len_q, seq_len_k)
              (only if return_weights=True)
    """
    if Q.ndim != 4 or K.ndim != 4 or V.ndim != 4:
        raise ValueError(
            f"Q, K, V must be 4D (batch, heads, seq_len, d_k); "
            f"got shapes {Q.shape}, {K.shape}, {V.shape}"
        )

    d_k = Q.shape[-1]

    # --- Step 1: raw attention scores, QK^T ---------------------------
    # Q: (batch, heads, seq_q, d_k)
    # K: (batch, heads, seq_k, d_k)  -> transpose last two dims -> (batch, heads, d_k, seq_k)
    # matmul broadcasts over the leading (batch, heads) axes automatically.
    K_T = np.swapaxes(K, -1, -2)                 # (batch, heads, d_k, seq_k)
    scores = np.matmul(Q, K_T)                   # (batch, heads, seq_q, seq_k)

    # --- Step 2: scale by sqrt(d_k) ------------------------------------
    # Prevents dot products from growing large in magnitude as d_k grows,
    # which would otherwise push softmax into saturated (near-zero-gradient)
    # regions.
    scores = scores / np.sqrt(d_k)

    # --- Step 3: apply mask (e.g. causal) ------------------------------
    if mask is not None:
        scores = scores + mask  # broadcasts (seq_q, seq_k) -> (batch, heads, seq_q, seq_k)

    # --- Step 4: softmax over the key dimension ------------------------
    weights = softmax(scores, axis=-1)            # (batch, heads, seq_q, seq_k)

    # --- Step 5: weighted sum of values ---------------------------------
    output = np.matmul(weights, V)                # (batch, heads, seq_q, d_v)

    if return_weights:
        return output, weights
    return output


# ----------------------------------------------------------------------
# 4. Helpers to go from (batch, seq_len, d_model) <-> multi-head 4D form
# ----------------------------------------------------------------------
def split_heads(x: np.ndarray, num_heads: int) -> np.ndarray:
    """
    (batch, seq_len, d_model) -> (batch, num_heads, seq_len, d_k)
    where d_k = d_model / num_heads.
    """
    batch, seq_len, d_model = x.shape
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")
    d_k = d_model // num_heads
    x = x.reshape(batch, seq_len, num_heads, d_k)
    return np.transpose(x, (0, 2, 1, 3))  # -> (batch, num_heads, seq_len, d_k)


def combine_heads(x: np.ndarray) -> np.ndarray:
    """
    (batch, num_heads, seq_len, d_k) -> (batch, seq_len, d_model)
    """
    batch, num_heads, seq_len, d_k = x.shape
    x = np.transpose(x, (0, 2, 1, 3))  # -> (batch, seq_len, num_heads, d_k)
    return x.reshape(batch, seq_len, num_heads * d_k)


# ----------------------------------------------------------------------
# 5. Multi-head attention wrapper (still pure NumPy, no nn.Linear etc.)
# ----------------------------------------------------------------------
def multi_head_attention(x: np.ndarray,
                          num_heads: int,
                          Wq: np.ndarray, Wk: np.ndarray, Wv: np.ndarray, Wo: np.ndarray,
                          causal: bool = False):
    """
    Full multi-head self-attention block built only from the primitives above.

    x  : (batch, seq_len, d_model)
    Wq, Wk, Wv, Wo : (d_model, d_model) projection matrices
    causal : whether to apply a causal mask (decoder-style)

    Returns
    -------
    out : (batch, seq_len, d_model)
    attn_weights : (batch, num_heads, seq_len, seq_len)
    """
    batch, seq_len, d_model = x.shape

    # Linear projections (plain matmul, no framework layers)
    Q = x @ Wq   # (batch, seq_len, d_model)
    K = x @ Wk
    V = x @ Wv

    # Split into heads -> 4D tensors
    Qh = split_heads(Q, num_heads)   # (batch, heads, seq_len, d_k)
    Kh = split_heads(K, num_heads)
    Vh = split_heads(V, num_heads)

    mask = generate_causal_mask(seq_len) if causal else None

    out_h, attn_weights = scaled_dot_product_attention(
        Qh, Kh, Vh, mask=mask, return_weights=True
    )

    out = combine_heads(out_h)   # (batch, seq_len, d_model)
    out = out @ Wo                # final output projection

    return out, attn_weights


# ----------------------------------------------------------------------
# 6. Self-tests / demonstration
# ----------------------------------------------------------------------
if __name__ == "__main__":
    np.random.seed(42)

    # ---- Test A: shape correctness for batched, multi-head input ----
    batch, num_heads, seq_len, d_k = 2, 4, 6, 8
    Q = np.random.randn(batch, num_heads, seq_len, d_k)
    K = np.random.randn(batch, num_heads, seq_len, d_k)
    V = np.random.randn(batch, num_heads, seq_len, d_k)

    out, weights = scaled_dot_product_attention(Q, K, V, return_weights=True)
    assert out.shape == (batch, num_heads, seq_len, d_k)
    assert weights.shape == (batch, num_heads, seq_len, seq_len)
    # every row of the attention matrix must sum to 1 (valid probability dist.)
    row_sums = weights.sum(axis=-1)
    assert np.allclose(row_sums, 1.0), "softmax rows must sum to 1"
    print("[Test A] Batched multi-head shapes & softmax normalization: PASS")

    # ---- Test B: causal mask blocks future positions -----------------
    mask = generate_causal_mask(seq_len)
    out_c, weights_c = scaled_dot_product_attention(Q, K, V, mask=mask, return_weights=True)
    # For query position i, weight on key position j>i must be ~0
    future_leak = 0.0
    for i in range(seq_len):
        for j in range(i + 1, seq_len):
            future_leak += weights_c[:, :, i, j].sum()
    assert np.isclose(future_leak, 0.0, atol=1e-6), "causal mask leaked into the future"
    print("[Test B] Causal masking blocks future tokens: PASS")

    # ---- Test C: manual (non-vectorized) reference check --------------
    # Recompute attention for a single (batch=0, head=0) slice by hand,
    # with explicit Python loops, and compare to the vectorized result.
    def manual_attention_single(q, k, v, use_mask=False):
        n, d = q.shape
        scores = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                scores[i, j] = np.dot(q[i], k[j]) / np.sqrt(d)
                if use_mask and j > i:
                    scores[i, j] = -np.inf
        # row-wise softmax
        w = np.zeros_like(scores)
        for i in range(n):
            row = scores[i]
            row = row - np.max(row)
            e = np.exp(row)
            w[i] = e / e.sum()
        out = w @ v
        return out, w

    q0, k0, v0 = Q[0, 0], K[0, 0], V[0, 0]
    manual_out, manual_w = manual_attention_single(q0, k0, v0, use_mask=True)
    assert np.allclose(manual_out, out_c[0, 0], atol=1e-6)
    assert np.allclose(manual_w, weights_c[0, 0], atol=1e-6)
    print("[Test C] Vectorized result matches manual per-element reference: PASS")

    # ---- Test D: end-to-end multi-head attention block ----------------
    d_model = 32
    heads = 4
    seq = 10
    batch2 = 3
    x = np.random.randn(batch2, seq, d_model)
    Wq = np.random.randn(d_model, d_model) * 0.1
    Wk = np.random.randn(d_model, d_model) * 0.1
    Wv = np.random.randn(d_model, d_model) * 0.1
    Wo = np.random.randn(d_model, d_model) * 0.1

    out_mha, attn_w = multi_head_attention(x, heads, Wq, Wk, Wv, Wo, causal=True)
    assert out_mha.shape == (batch2, seq, d_model)
    assert attn_w.shape == (batch2, heads, seq, seq)
    print("[Test D] End-to-end causal multi-head attention block: PASS")

    print("\nAll tests passed.")

[Test A] Batched multi-head shapes & softmax normalization: PASS
[Test B] Causal masking blocks future tokens: PASS
[Test C] Vectorized result matches manual per-element reference: PASS
[Test D] End-to-end causal multi-head attention block: PASS

All tests passed.
